In [1]:
import os
import json
import glob

DATASET_ROOT = "Datasets/FUNSD_Dataset/dataset" # root folder

# Output files
TRAIN_OUTPUT = "phase1_funsd_train.jsonl"
TEST_OUTPUT = "phase1_funsd_test.jsonl"

In [2]:
def parse_funsd_json(json_path):
    """
    Parses a FUNSD annotation file (which acts as the OCR output).
    Returns:
      - input_context: Text representation of the document layout.
      - gold_kv: Extracted Key-Value pairs.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # extract form items
    form_items = data.get('form', [])
    item_map = {item['id']: item for item in form_items}
    
    # Create the "Input Context" (annotated ocr so simulate ocr outputs)
    # We include the [x1, y1, x2, y2] box so the SLM learns layout
    context_parts = []

    # loops through all form items
    for item in form_items:

        # extarcting 
        text = item.get('text', '').strip()
        box = item.get('box', [])
        if text:
            context_parts.append(f"{text} {box}")
    
    # joining all parts
    input_context = " || ".join(context_parts)
    
    # Extract Key-Value Pairs (Gold Labels)
    '''
    Golden key-value pairs are the ground-truth question–answer mappings
    manually annotated in a dataset and used as correct labels for training
    and evaluation.
    '''
    
    kv_pairs = {} # empty dictionaries to store Q n A
    for item in form_items:

        # Selecting only the questions Ex - Name, Age....
        if item.get('label') == 'question':
            question_text = item.get('text', '').strip()
            links = item.get('linking', [])
            answer_texts = []
            
            for link in links:
                try:
                    # link is [id1, id2]. Find the ID that isnt the current one.
                    other_id = link[1] if link[0] == item['id'] else link[0]
                    linked_item = item_map.get(other_id)
                    if linked_item and linked_item.get('label') == 'answer':
                        answer_texts.append(linked_item.get('text', '').strip())
                except Exception:
                    continue

            # combaining QnA
            if answer_texts:
                full_answer = " ".join(answer_texts)
                kv_pairs[question_text] = full_answer

    return input_context, json.dumps(kv_pairs, ensure_ascii=False)

In [3]:
def process_funsd_folder(folder_name, output_file):
    """
    Process a specific subfolder (e.g., 'training_data' or 'testing_data')
    """
    target_path = os.path.join(DATASET_ROOT, folder_name, "annotations")
    
    print(f"Scanning {target_path}...")

    # finds the folders
    if not os.path.exists(target_path):
        print(f"  [!] Warning: Folder not found: {target_path}")
        return 0

    # finds all the json files
    json_files = glob.glob(os.path.join(target_path, "*.json"))
    print(f"  Found {len(json_files)} files. Processing...")

    processed_count = 0
    with open(output_file, 'w', encoding='utf-8') as f_out:
        for j_path in json_files:
            try:

                # processes each file
                ctx, gold = parse_funsd_json(j_path)
                
                # skips empty forms
                if ctx and gold != "{}":
                    entry = {
                        "source": "FUNSD",
                        "split": folder_name, # 'training_data' or 'testing_data'
                        "id": os.path.basename(j_path).replace('.json', ''),
                        "input": ctx,
                        "output": gold
                    }
                    f_out.write(json.dumps(entry) + "\n")
                    processed_count += 1
            except Exception as e:
                print(f"  Error processing {os.path.basename(j_path)}: {e}")
                
    print(f"  Saved {processed_count} entries to {output_file}")
    return processed_count

In [4]:
print("--- Processing TRAINING Data ---")
train_count = process_funsd_folder("training_data", TRAIN_OUTPUT)

--- Processing TRAINING Data ---
Scanning Datasets/FUNSD_Dataset/dataset\training_data\annotations...
  Found 149 files. Processing...
  Saved 147 entries to phase1_funsd_train.jsonl


In [5]:
print("\n--- Processing TESTING Data ---")
test_count = process_funsd_folder("testing_data", TEST_OUTPUT)


--- Processing TESTING Data ---
Scanning Datasets/FUNSD_Dataset/dataset\testing_data\annotations...
  Found 50 files. Processing...
  Saved 47 entries to phase1_funsd_test.jsonl


In [6]:
print("\n" + "="*30)
print(f"Phase 1A Complete!")
print(f"Training Samples: {train_count} (Saved to {TRAIN_OUTPUT})")
print(f"Testing Samples:  {test_count}  (Saved to {TEST_OUTPUT})")


Phase 1A Complete!
Training Samples: 147 (Saved to phase1_funsd_train.jsonl)
Testing Samples:  47  (Saved to phase1_funsd_test.jsonl)
